# 03 · Variables por cliente (RFM extendido)

> **Objetivo:** construir el DataFrame por cliente que la Unidad 3 usa
> como entrada. La funcion `construir_rfm(ventas, fecha_corte)` es la
> misma que U3 reutiliza con cortes anteriores; aqui la llamamos con
> `FECHA_SNAPSHOT = max(InvoiceDate) + 1 dia`.

## Variables que vamos a dejar

- **recencia:** dias desde la ultima compra hasta el corte.
- **antiguedad:** dias desde la primera compra hasta el corte.
- **frecuencia:** facturas distintas.
- **monetario:** suma de `Importe`.
- **productos_distintos:** `StockCode` unicos.

In [ ]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from datetime import timedelta

import polars as pl
from segmentacion.config import PROCESSED_DATA_DIR
from segmentacion.data.cargar import leer_parquet
from segmentacion.features.rfm import construir_rfm

PARQUET_VENTAS = PROCESSED_DATA_DIR / "ventas_validas.parquet"
PARQUET_RFM = PROCESSED_DATA_DIR / "rfm_u2.parquet"
assert PARQUET_VENTAS.exists(), "Ejecuta `02_limpieza.ipynb` primero."
ventas = leer_parquet(PARQUET_VENTAS)
FECHA_SNAPSHOT = ventas["InvoiceDate"].max() + timedelta(days=1)
print(f"FECHA_SNAPSHOT: {FECHA_SNAPSHOT}")
print(f"ventas: {ventas.height:,}".replace(",", " "))

## 1. Construir RFM

In [ ]:
rfm = construir_rfm(ventas, FECHA_SNAPSHOT)
print(f"clientes: {rfm.height:,}".replace(",", " "))
rfm.head()

## 2. Distribuciones

Cola larga en `monetario` y `frecuencia`: ambos son candidatos a `log1p`
en U3.

In [ ]:
rfm.select(["recencia", "antiguedad", "frecuencia", "monetario", "productos_distintos"]).describe()

## 3. Asociacion frecuencia-monetario (comprobacion rapida)

In [ ]:
rho = rfm.select(pl.corr("frecuencia", "monetario", method="spearman")).item()
print(f"Spearman frecuencia-monetario: {rho:.3f}  (OVA: 0,858)")

## 4. Guardar para la Unidad 3

In [ ]:
rfm.write_parquet(PARQUET_RFM, compression="zstd")
print(f"escrito: {PARQUET_RFM} ({PARQUET_RFM.stat().st_size / 1e6:.1f} MB)")

## Preguntas de reflexion

1. `construir_rfm` usa `InvoiceDate < fecha_corte` (estrictamente menor).
   ¿Que riesgo evita ese `<` cuando U3 defina cortes para evaluar
   recompra 90 dias en el futuro?
2. ¿Deberia `antiguedad` entrar al pipeline sklearn de U3 como feature
   crudo, o tiene sentido convertirlo a razon (`antiguedad / 365`)?
3. ¿Que indica la fuerte Spearman frecuencia-monetario para la
   seleccion de features? ¿Podrian estar midiendo lo mismo?

> **Siguiente paso:** `05_eda_automatizado.ipynb` — perfil con
> `skimpy` y `fg-data-profiling` sobre una muestra.